# Notebook 3b: Diagnostics — more statistical power, and what the labels reward

**Before running:** Runtime → Change runtime type → **T4 GPU**. Run time about 10–15 minutes.

Notebook 3 left three open questions. This notebook answers them.

1. **Too few test jobs.** With 45 jobs the confidence intervals are wide, and nothing beat TF-IDF significantly. The **zero-shot** methods never see any labels, so we can evaluate them on **every usable job in the dataset (about 290)**. That gives about 6× more statistical power, with no leakage, because nothing is trained.
2. **Where do the methods succeed and fail?** Two per-job AUC scores (0.5 = chance, 1.0 = perfect):
   - **AUC fit-vs-no**: can the method separate *Good or Potential* from *No Fit*?
   - **AUC good-vs-potential**: among relevant candidates, can it tell *Good* from *Potential*?
3. **Do the labels depend on the resume more than on the job?** In this dataset the same resume appears with many jobs. A **resume-prior** baseline scores each candidate by that resume's *average label with other jobs*, without reading the job at all. If it scores well, the labels are partly a property of the resume. That would explain why fine-tuning looked good in the standard split but not in the strict one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers einops rank-bm25

import re, time, os, gc
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
device = 'cuda'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)]
jobs = {jid: grp.reset_index(drop=True) for jid, grp in evald.groupby('job_id')}
all_resumes = all_df.resume_text.unique().tolist()
print(f'{len(jobs)} usable jobs, {len(evald)} pairs, {evald.resume_id.nunique()} resumes')
print(evald.groupby('split').job_id.nunique().to_dict())

## 1. Metrics

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))

def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k])
    return dcg(r[:k]) / ideal if ideal > 0 else 0.0

def precision_at_k(r, k, min_rel=2):
    return np.mean([x >= min_rel for x in r[:k]])

def mrr(r, min_rel=2):
    for i, x in enumerate(r):
        if x >= min_rel:
            return 1.0 / (i + 1)
    return 0.0

def auc_or_nan(y, s):
    y = np.asarray(y)
    return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

def bootstrap_ci(values, n=2000, seed=SEED):
    v = np.asarray(values, dtype=float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(seed)
    means = [rng.choice(v, len(v), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

METRICS = ['NDCG@5', 'NDCG@10', 'P@5', 'MRR', 'AUC fit-vs-no', 'AUC good-vs-potential']

def per_job_metrics(name, scores):
    rng = np.random.default_rng(SEED)
    rows = []
    for jid, grp in jobs.items():
        s = np.asarray(scores[jid], dtype=float)
        rel = grp.relevance.to_numpy()
        ranked = rel[np.lexsort((rng.random(len(s)), -s))].tolist()
        fit = rel > 0
        rows.append({'method': name, 'job_id': jid, 'split': grp.split.iloc[0],
                     'NDCG@5': ndcg_at_k(ranked, 5), 'NDCG@10': ndcg_at_k(ranked, 10),
                     'P@5': precision_at_k(ranked, 5), 'MRR': mrr(ranked),
                     'AUC fit-vs-no': auc_or_nan(fit.astype(int), s),
                     'AUC good-vs-potential': auc_or_nan((rel[fit] == 2).astype(int), s[fit]) if fit.sum() >= 2 else np.nan})
    return pd.DataFrame(rows)

def summarise(pq):
    out = {'method': pq.method.iloc[0], 'jobs': len(pq)}
    for m in METRICS:
        lo, hi = bootstrap_ci(pq[m])
        out[m] = round(np.nanmean(pq[m]), 4)
        out[m + ' CI'] = f'[{lo:.3f}, {hi:.3f}]'
    return out

per_query = []
def run(name, score_fn):
    scores = {jid: score_fn(grp.job_text.iloc[0], grp) for jid, grp in jobs.items()}
    pq = per_job_metrics(name, scores)
    per_query.append(pq)
    s = summarise(pq)
    print(f"{name:32} NDCG@10 {s['NDCG@10']:.4f} {s['NDCG@10 CI']} | AUC fit/no {s['AUC fit-vs-no']:.3f} "
          f"| AUC good/pot {s['AUC good-vs-potential']:.3f} | P@5 {s['P@5']:.3f} | MRR {s['MRR']:.3f}")

## 2. Non-content baselines: Random and Resume-prior
**Resume-prior** never reads the job. It scores a candidate by the resume's mean label across **all other jobs** in the dataset (leave-one-job-out). Resumes seen with no other job get the global mean.

In [ ]:
_rng = np.random.default_rng(SEED)
run('Random', lambda job, grp: _rng.random(len(grp)))

tot = all_df.groupby('resume_id').relevance.agg(['sum', 'count'])
global_mean = all_df.relevance.mean()
def resume_prior(job, grp):
    out = []
    for rid, rel in zip(grp.resume_id, grp.relevance):
        s, c = tot.loc[rid, 'sum'] - rel, tot.loc[rid, 'count'] - 1
        out.append(s / c if c > 0 else global_mean)
    return np.array(out)
run('Resume-prior (no job text)', resume_prior)

## 3. Keyword baselines
TF-IDF weights are learned from all job and resume texts. This uses **no labels**, so it isn't leakage. BM25 statistics come from all resumes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(all_df.job_text.unique()) + all_resumes)
run('TF-IDF', lambda job, grp: cosine_similarity(tfidf.transform([job]), tfidf.transform(grp.resume_text)).ravel())

def tokenize(text):
    return [t for t in re.findall(r'[a-z0-9][a-z0-9+#.]*', text.lower()) if t not in ENGLISH_STOP_WORDS]
bm25 = BM25Okapi([tokenize(r) for r in all_resumes])
tok_cache = {r: tokenize(r) for r in all_resumes}
def bm25_scores(job, grp):
    q = set(tokenize(job)); out = []
    for r in grp.resume_text:
        doc = tok_cache[r]; tf = pd.Series(doc).value_counts(); dl = len(doc); s = 0.0
        for term in q:
            f = tf.get(term, 0)
            if f:
                idf = bm25.idf.get(term, np.log((bm25.corpus_size + 0.5) / 0.5))
                s += idf * f * (bm25.k1 + 1) / (f + bm25.k1 * (1 - bm25.b + bm25.b * dl / bm25.avgdl))
        out.append(s)
    return np.array(out)
run('BM25', bm25_scores)

## 4. Zero-shot embedding models (same settings as Notebook 3)

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def chunk_words(text, size=200, stride=150):
    w = text.split()
    return [text] if len(w) <= size else [' '.join(w[i:i + size]) for i in range(0, len(w) - size + stride, stride)]

def embedding_run(label, model, qp='', dp='', chunked=False, batch_size=32):
    enc = lambda texts, p: model.encode([p + t for t in texts], batch_size=batch_size, normalize_embeddings=True,
                                       convert_to_numpy=True, show_progress_bar=False)
    pieces = {r: (chunk_words(r) if chunked else [r]) for r in all_resumes}
    flat = [p for r in all_resumes for p in pieces[r]]
    emb = enc(flat, dp); index, i = {}, 0
    for r in all_resumes:
        n = len(pieces[r]); index[r] = emb[i:i + n]; i += n
    job_texts = [grp.job_text.iloc[0] for grp in jobs.values()]
    job_emb = dict(zip(job_texts, enc(job_texts, qp)))
    run(label, lambda job, grp: np.array([float((index[r] @ job_emb[job]).max()) for r in grp.resume_text]))

MODELS = [('MiniLM-L6', 'sentence-transformers/all-MiniLM-L6-v2', '', '', 256),
          ('MPNet-base', 'sentence-transformers/all-mpnet-base-v2', '', '', 384),
          ('BGE-base', 'BAAI/bge-base-en-v1.5', '', '', 512),
          ('E5-base', 'intfloat/e5-base-v2', 'query: ', 'passage: ', 512)]
for name, hf_id, qp, dp, max_len in MODELS:
    model = SentenceTransformer(hf_id, device=device); model.max_seq_length = max_len
    embedding_run(f'{name} (zero-shot)', model, qp, dp, False)
    embedding_run(f'{name} (zero-shot, chunked)', model, qp, dp, True)
    del model; gc.collect(); torch.cuda.empty_cache()

try:
    model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', device=device, trust_remote_code=True)
    model.max_seq_length = 2048
    embedding_run('Nomic-v1.5 (zero-shot, long)', model, 'search_query: ', 'search_document: ', False, batch_size=8)
    del model; gc.collect(); torch.cuda.empty_cache()
except Exception as e:
    print('Nomic skipped:', e)

ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device, max_length=512)
run('CrossEnc-MiniLM (zero-shot)', lambda job, grp: ce.predict([(job, r) for r in grp.resume_text], batch_size=32, show_progress_bar=False))
del ce; gc.collect(); torch.cuda.empty_cache()

## 5. Results on all usable jobs

In [ ]:
pq_all = pd.concat(per_query, ignore_index=True)
table = pd.DataFrame([summarise(pq) for pq in per_query]).sort_values('NDCG@10', ascending=False)
table[['method', 'jobs', 'NDCG@10', 'NDCG@10 CI', 'AUC fit-vs-no', 'AUC good-vs-potential', 'AUC good-vs-potential CI', 'P@5', 'MRR']]

## 6. Significance against TF-IDF (paired permutation, Holm-corrected)
Tested on NDCG@10 and on AUC good-vs-potential, the part the keyword methods seemed unable to do.

In [ ]:
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    d = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

def holm(p):
    order = np.argsort(p); k = len(p); adj = np.empty(k); running = 0
    for rank, i in enumerate(order):
        running = max(running, (k - rank) * p[i]); adj[i] = min(1, running)
    return adj

rows = []
for metric in ['NDCG@10', 'AUC good-vs-potential']:
    wide = pq_all.pivot(index='job_id', columns='method', values=metric)
    for m in wide.columns:
        if m == 'TF-IDF':
            continue
        both = wide[[m, 'TF-IDF']].dropna()
        rows.append({'metric': metric, 'method': m, 'jobs': len(both),
                     'diff vs TF-IDF': round((both[m] - both['TF-IDF']).mean(), 4),
                     'p': paired_permutation_p(both[m], both['TF-IDF'])})
sig = pd.DataFrame(rows)
sig['p (Holm)'] = np.concatenate([holm(g.p.to_numpy()) for _, g in sig.groupby('metric', sort=False)])
sig['significant'] = sig['p (Holm)'] < 0.05
sig.sort_values(['metric', 'diff vs TF-IDF'], ascending=[True, False])

## 7. Save

In [ ]:
table.to_csv(f'{RESULTS}/03b_all_jobs_summary.csv', index=False)
pq_all.to_csv(f'{RESULTS}/03b_all_jobs_per_query.csv', index=False)
sig.to_csv(f'{RESULTS}/03b_significance_vs_tfidf.csv', index=False)
print('saved')